# R Master v9 Fix4 · No-Render Geometry Compare

**Fix3 连续两次都死在 `[1/7] rendering front`，所以这版彻底取消 Blender/Eevee 渲染。**

Blender 只做一次轻量几何测量，然后退出；所有对比图由 Colab Python 直接画轮廓包络和宽/深曲线。这样更适合判断 Lap 能不能当 Mona 的 body donor，也避开手机端渲染导致的运行时断开。

不改源模型，不接头，不转权重，不烘 Rest Pose，不导 VRM。直接“全部运行”。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import shutil, subprocess, json, zipfile, math, os

ROOT=Path("/content/drive/MyDrive/R_Master")
CACHE=ROOT/"cache"
FIX3=ROOT/"v9_body_compare"/"fix3_latest"
OUT=ROOT/"v9_body_compare"/"fix4_latest"
OUT.mkdir(parents=True,exist_ok=True)
LOCAL=Path("/content/r_master_v9_fix4")
LOCAL.mkdir(parents=True,exist_ok=True)

STAGE=FIX3/"R_Master_v9_Fix3_COMPARE_STAGE.blend"
if not STAGE.exists() or STAGE.stat().st_size<20*1024*1024:
    raise RuntimeError("没找到 Fix3 compare stage。截图这一行给二蛋。")
print("✓ 直接复用 Fix3 stage：",round(STAGE.stat().st_size/1024/1024,1),"MiB")

BLENDER_VERSION="4.4.3"
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
BLENDER=BDIR/"blender"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
else:
    raise RuntimeError("v8 Blender Drive cache missing。截图这一行给二蛋。")

if not BLENDER.exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
if not BLENDER.exists(): raise FileNotFoundError(BLENDER)
print("✓ Blender ready")



In [ ]:
# Fix4: NO EEVEE / NO BLENDER IMAGE RENDER.
# Blender only opens the stage once, measures geometry, writes a small JSON, then exits.
PROFILE=OUT/"R_Master_v9_Fix4_profiles.json"
EXTRACT=LOCAL/"v9_fix4_extract.py"
EXTRACT.write_text(r'''
import bpy,sys,os,json,math
from mathutils import Vector

argv=sys.argv[sys.argv.index("--")+1:]
out=argv[argv.index("--out")+1]
body=bpy.data.objects.get("CMP_Mona")
lap=bpy.data.objects.get("CMP_Lap")
if not body or not lap:
    raise RuntimeError("CMP_Mona / CMP_Lap missing")

def world_points(o):
    M=o.matrix_world
    return [M@v.co for v in o.data.vertices]

P={"Mona":world_points(body),"Lap":world_points(lap)}

def bounds(q):
    mn=Vector((min(p.x for p in q),min(p.y for p in q),min(p.z for p in q)))
    mx=Vector((max(p.x for p in q),max(p.y for p in q),max(p.z for p in q)))
    return mn,mx

B={k:bounds(v) for k,v in P.items()}

# Normalize each mesh by its own height, floor at z=0, center X/Y.
def norm_points(name):
    q=P[name]; mn,mx=B[name]; h=mx.z-mn.z
    cx=(mn.x+mx.x)*.5; cy=(mn.y+mx.y)*.5
    return [((p.x-cx)/h,(p.y-cy)/h,(p.z-mn.z)/h) for p in q]

N={k:norm_points(k) for k in P}

def profile(q,angle_deg,bins=280):
    a=math.radians(angle_deg)
    ca,sa=math.cos(a),math.sin(a)
    rows=[]
    for i in range(bins):
        z0=i/bins; z1=(i+1)/bins
        us=[]
        for x,y,z in q:
            if z0<=z<z1 or (i==bins-1 and z==1.0):
                u=x*ca+y*sa
                us.append(u)
        if us:
            rows.append({"z":(z0+z1)*.5,"min":min(us),"max":max(us),"width":max(us)-min(us)})
    return rows

angles={"front":0.0,"three_quarter":45.0,"side":90.0}
profiles={name:{view:profile(q,a) for view,a in angles.items()} for name,q in N.items()}

# Key horizontal slab widths/depths in normalized coordinates.
levels={"head_neck":0.865,"shoulder":0.80,"chest":0.70,"waist":0.58,"pelvis":0.50,"upper_thigh":0.43}
def slab(q,zf,band=.012):
    pts=[p for p in q if abs(p[2]-zf)<=band]
    if not pts:return None
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
    return {"width":max(xs)-min(xs),"depth":max(ys)-min(ys),"count":len(pts)}

slabs={k:{n:slab(N[n],zf) for n in N} for k,zf in levels.items()}

# Difference metrics using matching z samples on front and side profiles.
def interp_width(rows,z):
    if not rows:return None
    r=min(rows,key=lambda r:abs(r["z"]-z))
    return r["width"]

zs=[0.34+i*(0.52/100) for i in range(101)]  # body-centric range, avoids feet/head noise
front_diffs=[]; side_diffs=[]
for z in zs:
    mw=interp_width(profiles["Mona"]["front"],z); lw=interp_width(profiles["Lap"]["front"],z)
    md=interp_width(profiles["Mona"]["side"],z); ld=interp_width(profiles["Lap"]["side"],z)
    if None not in (mw,lw): front_diffs.append(abs(mw-lw))
    if None not in (md,ld): side_diffs.append(abs(md-ld))
metrics={
 "mean_abs_front_width_diff_over_height":sum(front_diffs)/len(front_diffs) if front_diffs else None,
 "mean_abs_side_depth_diff_over_height":sum(side_diffs)/len(side_diffs) if side_diffs else None
}

report={
 "ok":True,
 "stage":"R_Master_v9_Fix4_NoRender_GeometryCompare",
 "method":"normalized vertex-envelope geometry profiles; no Blender image rendering",
 "vertices":{"Mona":len(P["Mona"]),"Lap":len(P["Lap"])},
 "profiles":profiles,
 "slabs":slabs,
 "metrics":metrics,
 "source_modified":False,
 "weights_transferred":False,
 "rest_pose_baked":False,
 "final_vrm":False
}
with open(os.path.join(out,"R_Master_v9_Fix4_profiles.json"),"w") as f:
    json.dump(report,f)
print("[v9fix4] PROFILE_OK",report["vertices"],report["metrics"],flush=True)
''',encoding="utf-8")

if PROFILE.exists() and PROFILE.stat().st_size>10000:
    print("✓ geometry profile checkpoint exists")
else:
    r=subprocess.run([str(BLENDER),"--background",str(STAGE),"--python",str(EXTRACT),
                      "--","--out",str(OUT)],
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print("\n".join([x for x in r.stdout.splitlines() if "[v9fix4]" in x or "Error" in x or "Traceback" in x])[-6000:])
    if r.returncode!=0:
        print(r.stdout[-12000:])
        raise RuntimeError("Fix4 geometry extraction failed")
print("✓ geometry extraction complete")



In [ ]:
# Pure Python/Matplotlib comparison: no Blender renderer.
import json, numpy as np, matplotlib.pyplot as plt
D=json.loads(PROFILE.read_text())
profiles=D["profiles"]

def outline_xy(rows):
    z=np.array([r["z"] for r in rows])
    lo=np.array([r["min"] for r in rows])
    hi=np.array([r["max"] for r in rows])
    return z,lo,hi

def save_outline(view,title,filename,zlim=(0,1)):
    fig,ax=plt.subplots(figsize=(5.4,8.0))
    for name,ls in [("Mona","-"),("Lap","--")]:
        z,lo,hi=outline_xy(profiles[name][view])
        m=(z>=zlim[0])&(z<=zlim[1])
        z=z[m];lo=lo[m];hi=hi[m]
        ax.plot(lo,z,ls,label=name)
        ax.plot(hi,z,ls)
    ax.axvline(0,linewidth=.8)
    ax.set_aspect("equal",adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("normalized horizontal profile")
    ax.set_ylabel("normalized height")
    ax.legend()
    ax.grid(alpha=.2)
    fig.tight_layout()
    p=OUT/filename;fig.savefig(p,dpi=170);plt.close(fig)
    return p

plots=[]
plots.append(save_outline("front","Front silhouette envelope","R_Master_v9_Fix4_front_profile.png"))
plots.append(save_outline("side","Side silhouette envelope","R_Master_v9_Fix4_side_profile.png"))
plots.append(save_outline("three_quarter","45° silhouette envelope","R_Master_v9_Fix4_three_quarter_profile.png"))
plots.append(save_outline("front","Chest / waist region","R_Master_v9_Fix4_chest_waist.png",(0.52,0.83)))
plots.append(save_outline("front","Pelvis / upper-thigh region","R_Master_v9_Fix4_pelvis_thigh.png",(0.34,0.64)))
plots.append(save_outline("front","Head / neck cut region","R_Master_v9_Fix4_head_neck.png",(0.78,1.0)))

# Width/depth curves.
fig,ax=plt.subplots(figsize=(6.2,7.2))
for name,ls in [("Mona","-"),("Lap","--")]:
    z=np.array([r["z"] for r in profiles[name]["front"]])
    w=np.array([r["width"] for r in profiles[name]["front"]])
    ax.plot(w,z,ls,label=name+" width")
    z2=np.array([r["z"] for r in profiles[name]["side"]])
    d=np.array([r["width"] for r in profiles[name]["side"]])
    ax.plot(d,z2,ls,label=name+" depth")
ax.set_title("Body width/depth profile vs height")
ax.set_xlabel("normalized span / body height")
ax.set_ylabel("normalized height")
ax.grid(alpha=.2);ax.legend();fig.tight_layout()
p=OUT/"R_Master_v9_Fix4_width_depth_curves.png";fig.savefig(p,dpi=170);plt.close(fig);plots.append(p)

print("✓ plots:")
for p in plots: print(" ",p.name)
print("\nKey metrics:",json.dumps(D["metrics"],indent=2))
print("\nSlabs:",json.dumps(D["slabs"],indent=2))



In [ ]:
from IPython.display import display,Image,Markdown
display(Markdown("## v9 Fix4 · Geometry Compare"))
for p in plots:
    display(Markdown("### "+p.stem.replace("R_Master_v9_Fix4_","").replace("_"," ")))
    display(Image(filename=str(p),width=460))

ZIP=OUT/"R_Master_v9_Fix4_Review.zip"
if ZIP.exists():ZIP.unlink()
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in plots:z.write(p,p.name)
    z.write(PROFILE,PROFILE.name)
print("✓ Review ZIP:",ZIP,round(ZIP.stat().st_size/1024/1024,2),"MiB")
from google.colab import files
files.download(str(ZIP))

